# Tarea 2

## Objetivo:

Implementar y contrastar dos modelos de clasificación explicativa —uno con features originales seleccionadas y otro con componentes principales— bajo criterios de rigor técnico y buenas prácticas de Data Mining.

### Sub-objetivos:
- Contrastar el impacto de la reducción de dimensionalidad mediante PCA frente al uso de features crudas en un modelo logístico
- Identificar y extraer los patrones clave de las features que determinan el comportamiento de la variable target.
- Eliminar la multicolinealidad del dataset mediante selección de features y PCA para obtener estimaciones de coeficientes más estables.
- Desarrollar un código limpio y estructurado, aplicando un manejo de errores eficiente para asegurar que el pipeline de datos no se rompa.
- Utilizar de forma autónoma la documentación de _scikit-learn_ y _statsmodels_ para implementar los algoritmos y resolver dudas técnicas.

## Tarea del estudiante:

Lee detenidamente las siguientes instrucciones.

El estudiante tendrá bastante libertad para realizar la tarea, utilizando sus propios criterios y estrategias.

- deberá estar limitado a las librerías de _pandas_, _numpy_, _pca_, _scikit-learn_ y _statsmodels_;
- no modificará los comentarios de los enunciados, pero sí podrá editar y crear celdas para completar cada una de las partes;
- en los apartados que se solicita "explicación" o "comentario", no olvidar dejar vuestra reflexión, si no estuviese, no se puntuaría ese apartado.

Una vez finalizada la actividad, guarda tu fichero, reinicia el kernel vuélvelo a ejecutar. No deben aparecer errores. Un notebook con errores es una Tarea incompleta. Para la entrega, TODO EL CÓDIGO DEBE ESTAR EJECUTADO (EN ORDEN) EN EL FICHERO QUE ENTREGÁIS.

RECUERDA SUBIR CADA UNO DE LOS FICHEROS .ipynb TAL CUAL (sueltos), SIN COMPRIMIR Y SIN CAMBIARLES EL NOMBRE. Los ficheros subidos deben tener exactamente el mismo nombre de fichero que tenían cuando los recibiste. No subas ningún PDF ni ningún fichero ZIP ni nada similar. La plataforma ya los separa automáticamente en carpetas que traen el nombre y apellidos del alumno, por lo que NO es necesario que lo pongas en ninguna parte.

## Evaluación:

- Este notebook se evalúa sobre 10 puntos.

# Parte 0: Librerías

In [ ]:
# Se reserva este espacio para la importación de librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pca import pca 
import seaborn as sns
sns.set_style('darkgrid')

import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

seed = 99

# el estudiante puede añadir aquí las librerías que necesite

# Parte 0: Dataset

A continuación, se da más detalle sobre las columnas:

**target**:
* _diagnosis_: M (maligno) o B (benigno), variable objetivo

**features**:
* _id_: identificador del paciente, sin valor predictivo
* _radius_mean_: radio medio de los núcleos celulares
* _texture_mean_: textura media (variación en escala de grises)
* _perimeter_mean_: perímetro medio de los núcleos
* _area_mean_: área media de los núcleos
* _smoothness_mean_: suavidad media del contorno
* _compactness_mean_: compacidad media (perimeter² / area - 1)
* _concavity_mean_: severidad media de las concavidades del contorno
* _concave_: ints_mean — número medio de puntos cóncavos
* _symmetry_mean_: simetría media de los núcleos
* _fractal_dimension_mean_: rugosidad media del contorno
* _radius_se_: error estándar del radio
* _texture_se_: error estándar de la textura
* _perimeter_se_: error estándar del perímetro
* _area_se_: error estándar del área
* _smoothness_se_: error estándar de la suavidad
* _compactness_se_: error estándar de la compacidad
* _concavity_se_: error estándar de la concavidad
* _concave_: ints_se — error estándar de los puntos cóncavos
* _symmetry_se_: error estándar de la simetría
* _fractal_dimension_se_: error estándar de la dimensión fractal
* _radius_worst_: radio medio de los 3 núcleos más grandes
* _texture_worst_: textura de los 3 núcleos más extremos
* _perimeter_worst_: perímetro de los 3 núcleos más extremos
* _area_worst_: área de los 3 núcleos más extremos
* _smoothness_worst_: suavidad de los 3 núcleos más extremos
* _compactness_worst_: compacidad de los 3 núcleos más extremos
* _concavity_worst_: concavidad de los 3 núcleos más extremos
* _concave_: ints_worst — puntos cóncavos de los 3 núcleos más extremos
* _symmetry_worst_: simetría de los 3 núcleos más extremos
* _fractal_dimension_worst_: dimensión fractal de los 3 núcleos más extremos

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/refs/heads/main/breast_cancer_data.csv")
df.head()

In [ ]:
df.shape

In [ ]:
# Todo el trabajo del estudiante se realizará con df1.
# Mantendremos la variable df para comparar cuando sea necesario.

df1 = df.copy()

# Parte 1: Estudio de correlaciones y multicolinealidad

## Correlaciones (0 puntos)
En este apartado se pide explorar las correlaciones de las features entre sí. No tratamos en este punto de transformar features ni crear nuevas, simplemente explorar en qué estado vienen.

In [ ]:
# Código del estudiante

df1.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(16,14))
sns.heatmap(df1.corr(numeric_only=True), annot=True, fmt=".2f")

plt.show()


## Multicolinealidad (0 puntos)
En este apartado se pide explorar la multicolinealidad de las features. No tratamos en este punto de transformar features ni crear nuevas, simplemente explorar en qué estado vienen.

In [ ]:
# Códido del estudiante

df1.isna().sum()

In [ ]:
# Código del estudiante

from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df1.drop(columns=["diagnosis", "id", "Unnamed: 32"])  
X =sm.add_constant(X) 


for ii, col in enumerate(X.columns):
    vif = variance_inflation_factor(X.values, ii) 
    print(f"{col}: {vif:.2f}")



# Parte 2: Modelo Explicativo con las features crudas

Dado que la muestra es pequeña, trataremos de hacer un estudio con todo el dataset. Sin train-test split a priori.

Así, se pedirá en este apartado realizar todo el análisis necesario exclusivamente con la librería de _statsmodels_ para ajustar un modelo explicativo de regresión logística. Para ello, es posible que tengáis que eliminar algunas features, modificar alguna existente, crear alguna nueva, trabajar outliers, plots, análisis, etc. 

Tenéis total libertad para elaborar el modelo explicativo con el conjunto de train según vuestro propio criterio. Todas las iteraciones y decisiones que toméis, estarán aquí reflejadas y comentadas por vuestra parte. Podéis crear todas las celdas que consideréis necesarias.

## Creación del modelo (2 puntos)

In [ ]:
# Código del estudiante


X = df1.drop(columns=["diagnosis", "id", "Unnamed: 32", "perimeter_mean", "area_mean", "perimeter_worst", "area_worst"])
X = sm.add_constant(X)
y = df1["diagnosis"].map({"M":1, "B":0})

model = sm.Logit(y, X)

resultado = model.fit()

resultado.summary()

In [ ]:
# Código del estudiante

X = df1[["radius_worst", "texture_worst","smoothness_worst", "compactness_worst", "concavity_worst", "concave points_worst",
    "symmetry_worst", "fractal_dimension_worst"]]

y = df1["diagnosis"].map({"M":1, "B":0})
X = sm.add_constant(X)

model = sm.Logit(y, X)

resultado = model.fit()

resultado.summary()

In [ ]:
# Código del estudiante

# En el primer intento se utilizó el conjunto completo de variables explicativas
# Sin embargo, debido a la elevada multicolinealidad detectada previamente mediante
# la matriz de correlaciones y el cálculo del VIF, el modelo presentaba problemas
# de estabilidad y no era capaz de estimar correctamente los coeficientes.

# Para mejorar la estabilidad e interpretabilidad del modelo logístico, se realiza
# una selección de variables manteniendo características representativas de cada
# grupo de información. Se utilizan inicialmente las variables "worst", ya que
# representan las medidas más extremas y suelen contener
# información relevante para diferenciar entre diagnósticos benignos y malignos.


In [ ]:
# Código del estudiante

X = df1[["radius_worst", "texture_worst", "smoothness_worst", "concavity_worst","concave points_worst"]]

y = df1["diagnosis"].map({"M":1, "B":0})

X = sm.add_constant(X)

model = sm.Logit(y, X)

resultado = model.fit()

resultado.summary()

Modelo de referencia. En la última de vuestras celdas dejad claro cuál es el modelo final de referencia a continuación.

In [ ]:
# Código del estudiante 

X = df1[["radius_worst", "texture_worst", "smoothness_worst", "concave points_worst"]]

y = df1["diagnosis"].map({"M":1, "B":0})
X = sm.add_constant(X)

model = sm.Logit(y, X)

resultado = model.fit()

resultado.summary()


In [ ]:

# Tras analizar las correlaciones y la multicolinealidad mediante VIF se observó
# una elevada redundancia entre las variables originales. La inclusión de todas
# las features provocaba problemas de estabilidad en la regresión logística.

# Por este motivo se realizó una selección progresiva de variables, manteniendo
# aquellas con mayor capacidad explicativa y eliminando variables redundantes o
# con poca significancia estadística.

# El modelo final está compuesto por cuatro variables:
#
# - radius_worst
# - texture_worst
# - smoothness_worst
# - concave points_worst



## Cross-validation (1 punto)

Este modelo cuenta con un conjunto de features y una target que han sido ajustados sin train-test split. Es un modelo explicativo y no sería necesario. Sin embargo, para estudiar la robustez en caso de que quisieramos convertirlo en algo predictivo, nos gustaría hacer una validación cruzada.

Para este sub-apartado, podemos utilizar la librería _sklearn_ y ajustar un modelo de Regresión Logística con las mismas features que el modelo de referencia que habéis seleccionado anteriormente. Solo queremos explorar la robustez, y para ello se pide aplicar Cross-Validation con 5 splits.

In [ ]:
# Código del estudiante

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression

X = df1[["radius_worst","texture_worst","smoothness_worst","concave points_worst"]]
y = df1["diagnosis"].map({"M":1, "B":0})

model = LogisticRegression(max_iter=1000)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)


print(scores)
print("Media AUC:", scores.mean())

## Conclusiones (1 punto)

Para finalizar este apartado, tendréis que aportar un resumen de las conclusiones de vuestro estudio. Se permite un máximo de 5 bullets.

In [ ]:
# Comentarios del estudiante

# Conclusiones:

# 1. El modelo funciona bien aunque cambiemos los datos utilizados en cada prueba.

# 2. El valor medio de AUC (0.983) indica que el modelo distingue muy bien entre tumores benignos y malignos.

# 3. Los resultados de las 5 particiones son parecidos, por lo que el modelo es estable.

# 4. El modelo no depende de una única división de los datos para obtener buenos resultados.

# 5. Cross Validation confirma que las variables seleccionadas generan un modelo fiable para realizar predicciones.

# Parte 3: Modelo Explicativo con PCA

Dado que la muestra es pequeña, trataremos de hacer un estudio con todo el dataset. Sin train-test split a priori.

## PCA (2 puntos)

Este trabajo lo haremos con un PCA previo, donde veremos si tiene sentido este trabajo para meterlo en un modelo. Primero veremos qué sale con las proyecciones a componentes principales. Examinaremos cómo se proyecta y veremos cómo se combina esta interpretación con la interpretación de un modelo explicativo.

Nota: Para trabajar PCA podéis usar la librería de _pca_ o de _sklearn_.

In [ ]:
# Código del estudiante

from sklearn.preprocessing import StandardScaler
X = df1.drop(columns=["diagnosis", "id", "Unnamed: 32"])

scaler = StandardScaler()

X_std = scaler.fit_transform(X)

In [ ]:
# Código del estudiante

from pca import pca

model = pca(n_components=13)

results = model.fit_transform(X_std)

results.keys()

In [ ]:
# Código del estudiante

df_pca = pd.DataFrame({
    'PC': [f'PC{i}' for i in range(1,14)],
    'explained_var': results['explained_var'],
    'variance_ratio': results['variance_ratio'],
}).round(2)
df_pca


## Creación del modelo (1.5 puntos)

Así, se pedirá en este apartado realizar todo el análisis necesario exclusivamente con la librería de _statsmodels_ para ajustar un modelo explicativo de regresión logística con las transformaciones de PCA realizadas en el apartado anterior. Para ello, es posible que tengáis que eliminar algunas features, modificar alguna existente, crear alguna nueva, trabajar outliers, plots, análisis, etc.

Tenéis total libertad para elaborar el modelo explicativo según vuestro propio criterio. Todas las iteraciones y decisiones que toméis, estarán aquí reflejadas y comentadas por vuestra parte. Podéis crear todas las celdas que consideréis necesarias.

In [ ]:
# Código del estudiante


X_pca = results["PC"].iloc[:, :5]
y = df1["diagnosis"].map({"M":1, "B":0})

X_pca = sm.add_constant(X_pca)

In [ ]:
modelo_pca = sm.Logit(y, X_pca)

resultado_pca = modelo_pca.fit()

resultado_pca.summary()

## Cross-validation (1 punto)

Este modelo cuenta con un conjunto de features y una target que han sido ajustados sin train-test split. Es un modelo explicativo y no sería necesario. Sin embargo, para estudiar la robustez en caso de que quisieramos convertirlo en algo predictivo, nos gustaría hacer una validación cruzada.

Para este sub-apartado, podemos utilizar la librería _sklearn_ y ajustar un modelo de Regresión Logística con las mismas features que el modelo de referencia que habéis seleccionado anteriormente. Solo queremos explorar la robustez, y para ello se pide aplicar Cross-Validation con 5 splits.

In [ ]:
# Código del estudiante

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression

X_pca = results["PC"].iloc[:, :5]
y = df1["diagnosis"].map({"M":1, "B":0})


model = LogisticRegression(max_iter=1000)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores_pca = cross_val_score(
    model,
    X_pca,
    y,
    cv=cv,
    scoring="roc_auc"
)


print(scores_pca)
print("Media AUC:", scores_pca.mean())

# Mejora del umbral de probabilidad (0.5 puntos)

Finalmente, se pedirá evaluar la métrica del modelo con todo el dataset con accuracy. Se pide explorar, en la siguiente celda, si se puede mejorar el umbral de probabilidad de la regresión logística para aumentar este valor.

In [ ]:
# Código del estudiante

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_pca = results["PC"].iloc[:, :5]
y = df1["diagnosis"].map({"M":1, "B":0})


model = LogisticRegression(max_iter=1000)

model.fit(X_pca, y)


probs = model.predict_proba(X_pca)[:,1]

resultados = []

for umbral in np.arange(0.1, 0.9, 0.05):
    pred = (probs >= umbral).astype(int)
    acc = accuracy_score(y, pred)
    resultados.append([umbral, acc])


df_umbral = pd.DataFrame(
    resultados,
    columns=["umbral", "accuracy"]
)


df_umbral.sort_values("accuracy", ascending=False)

## Conclusiones (1 punto)

Para finalizar este apartado, tendréis que aportar un resumen de las conclusiones de vuestro estudio. Se permite un máximo de 5 bullets.

En, al menos 1 bullet, tenéis que comentar obligatoriamente la interpretabilidad de cómo impacta una componente principal del modelo seleccionado en esta sección (ya que las componentes principales serán usadas como features del modelo seleccionado).

En, al menos 1 bullet, tenéis que explicar qué información proyecta cada componente principal (al menos de dos de ellas). En otras palabras, qué representa.

In [ ]:
# Comentarios del estudiante
# Conclusiones:

# 1.El análisis de correlaciones y VIF mostró una elevada multicolinealidad entre
#   las variables originales, especialmente entre las medidas relacionadas con tamaño celular (radio, perímetro y área).

# 2.El modelo con PCA obtuvo mejores resultados que el modelo con variables
#   originales, alcanzando un AUC medio de 0.9946 en Cross Validation frente
#   a 0.9833 del modelo sin PCA.

# 2.La componente principal PC1 es la más importante del modelo, ya que presenta
#   un coeficiente positivo elevado y significativo. Un aumento de PC1 incrementa
#   la probabilidad estimada de diagnóstico maligno.
#
# 4.PC1 recoge principalmente información relacionada con el tamaño y geometría
#   de los núcleos celulares (radio, perímetro y área), mientras que PC2 recoge
#   principalmente información relacionada con características de forma y textura
#   celular.
#
# 5.El ajuste del umbral de clasificación mostró que modificar ligeramente el
#   punto de corte de 0.5 a 0.45 o 0.55 mejora ligeramente la accuracy, aunque
#   el modelo ya presenta un rendimiento elevado con el umbral estándar.